# PDF Parsing Pipeline — Healthcare Documents with LiteParse

**Objective:** Extract structured text from healthcare PDF documents
(Explanation of Benefits, Lab Results, Claims) using LiteParse — fast,
local, no cloud dependency. Then feed parsed text into the LLM QA
assistant for natural language querying.

**Why this matters in healthcare data ops:**
- ~30% of healthcare data exists only in PDF documents
- Claims attachments, lab results, clinical reports often arrive as PDFs
- Manual data entry from PDFs is slow, error-prone, expensive
- LiteParse + LLM = automated extraction at scale, no PHI leaving the machine

**LiteParse advantages for HIPAA:**
- 100% local parsing — no cloud API calls
- PHI never leaves the local environment
- Bounding boxes enable field-level extraction (e.g., extract only
  member ID and date, discard the rest before LLM processing)

In [ ]:
import subprocess
import json
import os
from pathlib import Path

# Verify lit is installed
result = subprocess.run(["lit", "--version"], capture_output=True, text=True)
print(f"LiteParse: {result.stdout.strip()}")

# Test with a sample PDF
sample_pdf = "sample_pdfs/eob_sample.pdf"
if Path(sample_pdf).exists():
    print(f"Sample PDF found: {sample_pdf}")
else:
    print(f"No sample PDF at {sample_pdf} — will create synthetic example")

## 1. Create Synthetic Healthcare PDF Sample

For demonstration, generate a synthetic EOB (Explanation of Benefits)
document. In production, this would be a real received PDF.

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

os.makedirs("sample_pdfs", exist_ok=True)

# Synthetic EOB data
eo_b_data = [
    ["Field", "Value"],
    ["Member ID", "MEM000042"],
    ["Patient Name", "Sarah Chen"],
    ["Date of Birth", "1985-03-15"],
    ["Date of Service", "2026-03-10"],
    ["Provider", "Fort Wayne Medical Associates"],
    ["Service", "Office Visit — Established Patient"],
    ["CPT Code", "99214"],
    ["Billed Amount", "$285.00"],
    ["Allowed Amount", "$210.00"],
    ["Insurance Paid", "$168.00"],
    ["Patient Responsibility", "$42.00"],
    ["Plan Type", "PPO"],
    ["Metal Level", "Gold"],
    ["Claim ID", "CLM202603100042"],
]

doc = SimpleDocTemplate(
    "sample_pdfs/eob_sample.pdf",
    pagesize=letter,
    rightMargin=72, leftMargin=72,
    topMargin=72, bottomMargin=72,
)

styles = getSampleStyleSheet()
story = [
    Paragraph("EXPLANATION OF BENEFITS", styles["Title"]),
    Spacer(1, 0.3 * inch),
    Paragraph("Vital Health Insurance", styles["Heading2"]),
    Spacer(1, 0.2 * inch),
]

table = Table(eo_b_data, colWidths=[2.5 * inch, 4 * inch])
table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a3c5e')),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, -1), 10),
    ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor('#f5f8fa')),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f5f8fa')]),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#c0c0c0')),
    ('ALIGN', (0, 0), (0, -1), 'LEFT'),
    ('FONTNAME', (0, 1), (0, -1), 'Helvetica-Bold'),
    ('LEFTPADDING', (0, 0), (-1, -1), 8),
    ('RIGHTPADDING', (0, 0), (-1, -1), 8),
    ('TOPPADDING', (0, 0), (-1, -1), 5),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
]))

story.append(table)
doc.build(story)
print(f"✅ Synthetic EOB PDF created: sample_pdfs/eob_sample.pdf")

## 2. Parse with LiteParse

Run `lit parse` via subprocess. LiteParse extracts:
- Raw text per page
- Bounding boxes (bbox) with coordinates for each text block
- OCR layer (via Tesseract.js) for scanned documents

In [ ]:
# Parse the EOB PDF with LiteParse
import json
from pathlib import Path

os.makedirs("output", exist_ok=True)

result = subprocess.run(
    ["lit", "parse", "sample_pdfs/eob_sample.pdf", "--format", "json", "-o", "output/eob_parsed.json"],
    capture_output=True,
    text=True,
)

print(f"Return code: {result.returncode}")
if result.returncode != 0:
    print(f"STDERR: {result.stderr}")
else:
    print("✅ Parse successful")
    
    with open("output/eob_parsed.json") as f:
        parsed = json.load(f)
    
    print(f"\nTotal pages: {parsed.get('total_pages', 0)}")
    for page in parsed.get("pages", []):
        print(f"  Page {page['page_number']}: {len(page.get('text', ''))} chars, {len(page.get('blocks', []))} blocks")

## 3. Extract Specific Fields via Bounding Boxes

Use bounding box coordinates to extract only the fields you need.
This limits PHI exposure — only extract member ID, dates, amounts
rather than passing the full document to the LLM.

In [ ]:
from parse_pdfs import HealthcarePDFParser

parser = HealthcarePDFParser(ocr_enabled=True)
result = parser.parse("sample_pdfs/eob_sample.pdf")

print(f"Parsed: {result.total_pages} pages, {result.total_chars} chars\n")

# Print full text
for page in result.pages:
    print(f"--- Page {page.number} ---")
    print(page.text)

## 4. Field Extraction — Structured Output

Extract only the clinically/relevant fields from the EOB.
This mirrors what a real data ops pipeline would do:
PDF → LiteParse → Field Extraction → Structured Table → dbt

In [ ]:
import pandas as pd

# Parse again and extract structured fields
text = result.pages[0].text

# Simple field extraction by line parsing
# In production: use bbox coordinates for precision
fields = {}
for line in text.strip().split('\n'):
    if ':' in line:
        key, val = line.split(':', 1)
        fields[key.strip()] = val.strip()

print("Extracted fields:")
for k, v in fields.items():
    print(f"  {k}: {v}")

In [ ]:
# Convert to DataFrame for downstream processing
df = pd.DataFrame([fields])
df["_source_file"] = result.filename
df["_parsed_at"] = result.extracted_at

print("\nStructured table (ready for dbt / SQL):")
print(df.to_string(index=False))

# Export as CSV
df.to_csv("output/eob_structured.csv", index=False)
print("\nSaved: output/eob_structured.csv")

## 5. Feed to LLM QA Assistant

Use the parsed text as context for the LLM QA assistant.
This is a production pattern: PDF → LiteParse → LLM = natural language querying
of any PDF document without manual review.

In [ ]:
# Generate LLM prompt from parsed PDF
llm_prompt = parser.to_llm_prompt("sample_pdfs/eob_sample.pdf")

print("LLM prompt (first 1000 chars):")
print(llm_prompt[:1000])
print("...")

# In production: send this to eligibility_qa_chatbot.py
# python eligibility_qa_chatbot.py --query "what is the allowed amount for claim CLM202603100042?"

## 6. Screenshot Generation

Generate page screenshots for visual verification — useful when
reviewing parsing accuracy or when LLM agents need visual context.

In [ ]:
# Generate screenshots
import subprocess
os.makedirs("output/screenshots", exist_ok=True)

result_screenshot = subprocess.run(
    ["lit", "screenshot", "sample_pdfs/eob_sample.pdf",
     "-o", "output/screenshots", "--dpi", "150"],
    capture_output=True,
    text=True,
)

print(f"Screenshot return code: {result_screenshot.returncode}")
screenshots = list(Path("output/screenshots").glob("*.png"))
print(f"Screenshots generated: {[s.name for s in screenshots]}")

## 7. Batch Processing — Multiple PDFs

In production, a claims pipeline receives dozens of PDFs per day.
LiteParse handles batch directory processing efficiently.

In [ ]:
from parse_pdfs import LiteParseRunner

# Create a second sample (Lab Results)
lab_data = [
    ["LABORATORY RESULTS"],
    ["Patient: Marcus Johnson  |  DOB: 1990-07-22  |  MRN: MRN008819"],
    ["Collected: 2026-03-08  |  Reported: 2026-03-09"],
    ["Ordering Physician: Dr. Priya Sharma"],
    [""],
    ["Test", "Result", "Reference Range", "Flag"],
    ["Hemoglobin A1c", "6.2%", "< 5.7%", "HIGH"],
    ["Fasting Glucose", "118 mg/dL", "70-100 mg/dL", "HIGH"],
    ["Total Cholesterol", "215 mg/dL", "< 200 mg/dL", "HIGH"],
    ["LDL Cholesterol", "142 mg/dL", "< 100 mg/dL", "HIGH"],
    ["HDL Cholesterol", "38 mg/dL", "> 40 mg/dL", "LOW"],
    ["Triglycerides", "188 mg/dL", "< 150 mg/dL", "HIGH"],
]

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer

doc2 = SimpleDocTemplate("sample_pdfs/lab_results.pdf", pagesize=letter,
                         rightMargin=72, leftMargin=72, topMargin=72, bottomMargin=72)

styles2 = getSampleStyleSheet()
story2 = [Paragraph("LABORATORY RESULTS REPORT", styles2["Title"]), Spacer(1, 0.2*inch)]

for line in lab_data[1:4]:
    story2.append(Paragraph(line[0], styles2["Normal"]))
story2.append(Spacer(1, 0.1*inch))

t2 = Table(lab_data[4:], colWidths=[2.2*inch, 1.5*inch, 2*inch, 1*inch])
t2.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2e5e1a')),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, -1), 10),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#c0c0c0')),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f5f5f5')]),
    ('TEXTCOLOR', (3, 1), (3, -1), colors.HexColor('#cc0000')),  # Flag column in red
    ('FONTNAME', (3, 1), (3, -1), 'Helvetica-Bold'),
    ('ALIGN', (3, 0), (-1, -1), 'CENTER'),
    ('LEFTPADDING', (0, 0), (-1, -1), 8),
    ('TOPPADDING', (0, 0), (-1, -1), 5),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
]))

story2.append(t2)
doc2.build(story2)
print("✅ Lab results PDF created: sample_pdfs/lab_results.pdf")

In [ ]:
# Batch parse both PDFs
lit = LiteParseRunner()
batch_results = lit.batch_parse("sample_pdfs", "output/batch", output_format="json")

print("\nBatch results:")
for name, info in batch_results.items():
    status = "✅" if info["status"] == "success" else "❌"
    s = info.get("stats", {})
    print(f"  {status} {name}: {s.get('pages', '?')} pages, {s.get('chars', '?')} chars")

## Summary

This pipeline demonstrates:

| Step | Tool | Output |
|------|------|--------|
| PDF ingestion | LiteParse (`lit parse`) | Structured text + bboxes |
| Field extraction | HealthcarePDFParser | Named fields (member_id, amount...) |
| Screenshot verification | LiteParse (`lit screenshot`) | PNG images |
| Batch processing | LiteParse (`lit batch-parse`) | Directory of JSONs |
| LLM integration | `to_llm_prompt()` | Formatted text for QA chatbot |

**HIPAA note:** LiteParse runs 100% locally. PHI never leaves the machine.
Bbox-based field extraction lets you strip PHI before sending to LLM.

**Next steps for production:**
1. Connect output CSVs → dbt staging models (`stg_pdf_claims`)
2. Map extracted fields → OMOP concept IDs (CPT, LOINC, ICD-10)
3. Add to Prefect flow: PDF arrival in S3 → LiteParse → dbt run